# Task 3 — Fine-tuned vs. No-fine-tune Baseline

Compares the **fine-tuned** Llama-3.1-8B QLoRA run against the **un-fine-tuned base model** on Task 3 (Provision-Type Classification / LEDGAR), evaluated on the *same* 1,945-row validation set. The Task 1 / Task 2 counterparts are [finetune_vs_baseline_comparison.ipynb](finetune_vs_baseline_comparison.ipynb) and [task2_finetune_vs_baseline_comparison.ipynb](task2_finetune_vs_baseline_comparison.ipynb).

Inputs (both produced by Kaggle runs and downloaded via `kaggle/run.ps1`):

| Run | Notebook that produced it | Artifacts read here |
|---|---|---|
| Fine-tuned | `llm_fine_tuning_LORA_task3.ipynb` | `kaggle_output/eval_metrics.json` |
| Baseline (no fine-tune) | `llama_3.1_task_3_no_fine_tune.ipynb` | `kaggle_output_task3_baseline/no_finetune_baseline_task3/eval_metrics.json` |

The baseline notebook is an exact replica of the fine-tuning one with the training removed, and **runs the identical evaluation code** — same greedy generation, same `predict_label` parsing. Both build the validation set with the same `SEED=42` stratified sample of LexGLUE's official splits, so the comparison is apples-to-apples; a cell below verifies that from the saved JSONL.

### The four metrics (full order: [TASK3_NEXT_STEPS.md](docs/task_3/TASK3_NEXT_STEPS.md) step 3)

1. **Valid-label rate** — did the model answer with a bare label from the 100-label menu at all? (A base model that chats/explains/echoes fails this gate; every such answer is a wrong prediction.)
2. **Accuracy** — fraction where the predicted label == gold label.
3. **Macro-F1** — the **headline** number; LEDGAR is 137× imbalanced (EDA Finding 1), so the unweighted per-label mean is what matters.
4. **Micro-F1** — for LexGLUE-leaderboard comparison (published BERT-class yardstick ~87–88 micro / ~82 macro).

Structure:
1. **Setup & loaders** — paths, palette, load both metric files, sanity checks.
2. **Same-validation-set proof** — record-by-record JSONL comparison.
3. **Headline comparison** — the four metrics, with deltas.
4. **Per-label breakdown** — where the 100 per-label F1 scores moved.
5. **Confusions** — the label pairs each model trips on.
6. **Takeaways** — computed from the numbers, so they stay correct if you point at other runs.

## 1. Setup & loaders

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Configuration: point these at any two runs with the standard Task 3 layout ----
FINETUNED_DIR = Path("kaggle_output")                                              # fine-tuned run
BASELINE_DIR  = Path("kaggle_output_task3_baseline/no_finetune_baseline_task3")    # no-fine-tune baseline

# ---- Palette: one fixed color per model, everywhere in this notebook ----
# Same validated pair as the Task 1/2 comparison notebooks, so all three read as one system.
BASE_COLOR = "#1baf7a"   # baseline (aqua)
FT_COLOR   = "#2a78d6"   # fine-tuned (blue)
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
SURFACE = "#fcfcfb"

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.grid": True, "grid.color": GRID,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.labelcolor": INK, "text.color": INK,
})

assert FINETUNED_DIR.exists(), f"Fine-tuned run dir not found: {FINETUNED_DIR.resolve()}"
assert BASELINE_DIR.exists(), f"Baseline run dir not found: {BASELINE_DIR.resolve()}"
print("Fine-tuned run:", FINETUNED_DIR.resolve())
print("Baseline run:  ", BASELINE_DIR.resolve())

In [ ]:
def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


base_metrics = load_json(BASELINE_DIR / "eval_metrics.json")
ft_metrics = load_json(FINETUNED_DIR / "eval_metrics.json")

# Fixed display order: baseline first, fine-tuned second — reused by every chart.
RUNS = {
    "Baseline (no fine-tune)": base_metrics,
    "Fine-tuned": ft_metrics,
}
RUN_COLORS = {"Baseline (no fine-tune)": BASE_COLOR, "Fine-tuned": FT_COLOR}

# The four headline metrics, in reporting order (see the header).
METRICS = ["valid_label_rate", "accuracy", "macro_f1", "micro_f1"]
METRIC_LABELS = {
    "valid_label_rate": "Valid-label rate",
    "accuracy": "Accuracy",
    "macro_f1": "Macro-F1",
    "micro_f1": "Micro-F1",
}

# Canonical label order = the id order both eval files wrote per_label in.
LABELS = list(ft_metrics["per_label"].keys())

# --- Sanity checks -----------------------------------------------------------
# Both runs must be Task 3, and must have been evaluated on the same validation set.
for name, m in RUNS.items():
    assert m.get("task") == "task3_provision_classification", f"{name} is not a Task 3 run: {m.get('task')!r}"

assert base_metrics["n_validation_examples"] == ft_metrics["n_validation_examples"], (
    f"Different validation sizes: baseline={base_metrics['n_validation_examples']} "
    f"vs fine-tuned={ft_metrics['n_validation_examples']} — the runs are NOT comparable.")

# Per-label support is a property of the validation set, so it must match label-for-label.
base_support = {l: base_metrics["per_label"][l]["support"] for l in LABELS}
ft_support = {l: ft_metrics["per_label"][l]["support"] for l in LABELS}
assert base_support == ft_support, "Per-label support differs — the runs saw different validation data."

print("OK — both runs are Task 3, same size, identical per-label support.\n")
for name, m in RUNS.items():
    tuned = "no" if m.get("fine_tuned") is False else "yes"
    print(f"{name:>24}: {m['model_name']:<32} fine-tuned={tuned}  "
          f"({m['n_validation_examples']} validation examples, {len(LABELS)} labels)")

## 2. Same-validation-set proof

The size and per-label support matching is necessary but not sufficient. Each run also saves the exact JSONL it evaluated on, so compare them **record by record** — if these differ, every number below is meaningless.

In [ ]:
def load_val_records(run_dir):
    path = run_dir / "ledgar" / "validation" / "ledgar_task3_validation.jsonl"
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return sorted(json.dumps(json.loads(line), sort_keys=True)
                      for line in f if line.strip())


base_val = load_val_records(BASELINE_DIR)
ft_val = load_val_records(FINETUNED_DIR)

if base_val is None or ft_val is None:
    print("WARNING: a validation JSONL is missing; falling back on the "
          "size/support checks above.")
elif base_val == ft_val:
    print(f"OK — both runs evaluated the identical {len(base_val)} validation examples.")
else:
    overlap = len(set(base_val) & set(ft_val))
    raise AssertionError(
        f"Validation sets DIFFER (only {overlap}/{len(base_val)} examples shared) — "
        "the metric comparison below would not be apples-to-apples.")

## 3. Headline comparison

The four metrics overall. `Δ` is fine-tuned minus baseline, in percentage points. Read them as a gate: an answer that isn't a valid label can't be correct, so a low **valid-label rate** caps accuracy and both F1s from above.

In [ ]:
summary = pd.DataFrame(
    {name: {METRIC_LABELS[k]: m[k] for k in METRICS} for name, m in RUNS.items()}
).T
summary.loc["Δ (fine-tuned − baseline)"] = (
    summary.loc["Fine-tuned"] - summary.loc["Baseline (no fine-tune)"])

display(
    summary.style
    .format("{:.1%}", subset=(list(RUNS), slice(None)))
    .format("{:+.1f} pp", subset=(["Δ (fine-tuned − baseline)"], slice(None)))
    .set_caption("Task 3 overall metrics on the shared validation set")
)

In [ ]:
def bar_labels(ax, fmt="{:.0%}"):
    """Annotate each bar with its height, in text ink (never the series color)."""
    for p in ax.patches:
        h = p.get_height()
        ax.annotate(fmt.format(h), (p.get_x() + p.get_width() / 2, h),
                    ha="center", va="bottom", fontsize=9, color=INK)


x = np.arange(len(METRICS))
w = 0.38   # leaves a visible surface gap between the paired bars

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.set_axisbelow(True)   # grid behind the bars, never drawn across them
for i, (name, m) in enumerate(RUNS.items()):
    ax.bar(x + (i - 0.5) * (w + 0.02), [m[k] for k in METRICS],
           width=w, color=RUN_COLORS[name], label=name)

ax.set_xticks(x, [METRIC_LABELS[k] for k in METRICS])
ax.set_ylim(0, 1.15)
ax.set_ylabel("score")
ax.set_title("Task 3 overall — baseline vs. fine-tuned")
ax.legend(frameon=False, loc="upper right")
bar_labels(ax)
plt.tight_layout()
plt.show()

print("Valid-label rate is the first gate: an answer that is not one of the 100 labels\n"
      "is a wrong prediction, so it caps accuracy and both F1 scores from above.")

## 4. Per-label breakdown

Macro-F1 is the unweighted mean of 100 per-label F1 scores, so the headline number hides *where* the model improved. The scatter plots every label's baseline F1 against its fine-tuned F1: points on the diagonal did not move, points **above** it improved. The dumbbells then name the biggest gains and the labels where the fine-tuned model is still weakest (support in the label, since a low F1 over 3 examples is weaker evidence than over 20).

In [ ]:
def f1_of(m, label):
    return m["per_label"][label]["f1"]

per_label = pd.DataFrame({
    "support":   [ft_support[l] for l in LABELS],
    "base_f1":   [f1_of(base_metrics, l) for l in LABELS],
    "ft_f1":     [f1_of(ft_metrics, l) for l in LABELS],
}, index=LABELS)
per_label["delta"] = per_label["ft_f1"] - per_label["base_f1"]

# Scatter: baseline F1 (x) vs fine-tuned F1 (y). Above the diagonal = improved.
fig, ax = plt.subplots(figsize=(6.4, 6.4))
ax.set_axisbelow(True)
ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls="--", zorder=1)
ax.scatter(per_label["base_f1"], per_label["ft_f1"], s=34, color=FT_COLOR,
           edgecolor=SURFACE, linewidth=0.6, alpha=0.85, zorder=3)
ax.set_xlim(-0.03, 1.03)
ax.set_ylim(-0.03, 1.03)
ax.set_xlabel("Baseline per-label F1")
ax.set_ylabel("Fine-tuned per-label F1")
ax.set_title("Every label moved up and to the left\n(above the dashed line = fine-tuning improved it)")
ax.annotate("above = fine-tune wins", (0.04, 0.9), fontsize=8.5, color=MUTED, style="italic")
plt.tight_layout()
plt.show()

improved = int((per_label["delta"] > 1e-9).sum())
unchanged = int((per_label["delta"].abs() <= 1e-9).sum())
regressed = int((per_label["delta"] < -1e-9).sum())
print(f"Of {len(LABELS)} labels: {improved} improved, {unchanged} unchanged, {regressed} regressed after fine-tuning.")

In [ ]:
def dumbbell(ax, frame, title):
    y = np.arange(len(frame))
    for j, (label, r) in enumerate(frame.iterrows()):
        ax.plot([r["base_f1"], r["ft_f1"]], [j, j], color=GRID, lw=3,
                zorder=1, solid_capstyle="round")
        ax.scatter([r["base_f1"]], [j], s=70, color=BASE_COLOR, zorder=3,
                   edgecolor=SURFACE, linewidth=1.3)
        ax.scatter([r["ft_f1"]], [j], s=70, color=FT_COLOR, zorder=3,
                   edgecolor=SURFACE, linewidth=1.3)
    ax.set_yticks(y, [f"{l}  (n={frame.loc[l, 'support']})" for l in frame.index])
    ax.set_xlim(-0.05, 1.08)
    ax.set_xlabel("F1")
    ax.set_title(title)
    ax.grid(axis="y", visible=False)
    ax.invert_yaxis()

top_gains = per_label.sort_values("delta", ascending=False).head(15)
# Remaining weaknesses: lowest fine-tuned F1 among labels that actually appear in val.
still_weak = (per_label[per_label["support"] > 0]
              .sort_values("ft_f1").head(15))

fig, axes = plt.subplots(1, 2, figsize=(14, 6.2))
dumbbell(axes[0], top_gains, "Biggest F1 gains from fine-tuning")
dumbbell(axes[1], still_weak, "Where the fine-tuned model is still weakest")

axes[0].scatter([], [], s=70, color=BASE_COLOR, label="Baseline (no fine-tune)")
axes[0].scatter([], [], s=70, color=FT_COLOR, label="Fine-tuned")
fig.legend(*axes[0].get_legend_handles_labels(), frameon=False, ncol=2,
           loc="upper center", bbox_to_anchor=(0.5, 0.99))
fig.suptitle("Per-label F1: baseline dot -> fine-tuned dot (connector length = the change)", y=1.02)
plt.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

## 5. Confusions

Each run saved its top gold→pred confusions. The baseline's list is dominated by the invalid sentinel (`__INVALID__`) when the model would not answer with a bare label at all; the fine-tuned list surfaces the genuine *look-alikes* — the pairs worth a second look in the label design (Governing Laws/Jurisdictions, Assigns/Successors, Amendments/Modifications, Waivers/No Waivers).

In [ ]:
def conf_frame(m, k=15):
    rows = [(c["gold"], c["pred"], c["count"]) for c in m["top_confusions"][:k]]
    return pd.DataFrame(rows, columns=["gold", "pred", "count"])

base_conf = conf_frame(base_metrics)
ft_conf = conf_frame(ft_metrics)

# How much of the baseline's error mass is pure "wouldn't emit a valid label"?
base_invalid = sum(c["count"] for c in base_metrics["top_confusions"] if c["pred"] == "__INVALID__")
ft_invalid = sum(c["count"] for c in ft_metrics["top_confusions"] if c["pred"] == "__INVALID__")

print(f"Baseline top-confusion mass that is __INVALID__ (no valid label emitted): {base_invalid}")
print(f"Fine-tuned same: {ft_invalid}\n")

print("=== Baseline — top gold -> pred confusions ===")
display(base_conf)
print("=== Fine-tuned — top gold -> pred confusions ===")
display(ft_conf)

## 6. Takeaways

Computed from the loaded metrics, so this stays correct when the run directories point elsewhere.

In [ ]:
n = ft_metrics["n_validation_examples"]
print(f"Validation set: {n} examples across {len(LABELS)} provision-type labels\n")

for k in METRICS:
    b, f = base_metrics[k], ft_metrics[k]
    print(f"  {METRIC_LABELS[k]:<17} {b:>6.1%}  ->  {f:>6.1%}   ({(f - b) * 100:+.1f} pp)")

bvl, fvl = base_metrics["valid_label_rate"], ft_metrics["valid_label_rate"]
print(f"\n1. The FORMAT gate first. Valid-label rate {bvl:.0%} -> {fvl:.0%}: the base model "
      f"fails to answer\n   with a bare menu label on ~{1 - bvl:.0%} of examples, and every such answer is a "
      "wrong\n   prediction by definition — capping its accuracy and both F1 scores from above.")

print(f"\n2. HEADLINE (macro-F1, imbalance-aware): {base_metrics['macro_f1']:.1%} -> "
      f"{ft_metrics['macro_f1']:.1%} "
      f"({(ft_metrics['macro_f1'] - base_metrics['macro_f1']) * 100:+.1f} pp).")

lex_micro, lex_macro = 0.88, 0.82
print(f"\n3. vs. the LexGLUE BERT-class yardstick (~{lex_micro:.0%} micro / ~{lex_macro:.0%} macro): "
      f"fine-tuned reaches\n   {ft_metrics['micro_f1']:.0%} micro / {ft_metrics['macro_f1']:.0%} macro — "
      "a decoder-only model at a 100/label stratified sample, not the full split.")

improved = int((per_label["delta"] > 1e-9).sum())
regressed = int((per_label["delta"] < -1e-9).sum())
print(f"\n4. Per label: {improved}/{len(LABELS)} labels improved, {regressed} regressed. "
      f"Biggest single gain: '{per_label['delta'].idxmax()}' "
      f"({per_label['delta'].max() * 100:+.0f} pp).")

still_weak = per_label[per_label["support"] > 0].sort_values("ft_f1").head(3)
weak_str = ", ".join(f"{l} (F1 {r.ft_f1:.0%}, n={int(r.support)})"
                     for l, r in still_weak.iterrows())
print(f"\n5. Hardest labels after fine-tuning: {weak_str}.\n"
      "   These are where label design / more examples would help next — see the confusion pairs above.")